In [ ]:
import os
import numpy as np
import qiskit
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
from qiskit.visualization import plot_bloch_multivector
from qiskit.quantum_info import Statevector
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler

token = os.getenv('IBM_QUANTUM_TOKEN')
instance = os.getenv('IBM_QUANTUM_INSTANCE')

try:
    QiskitRuntimeService.save_account(
        channel="ibm_quantum_platform",
        token=token,
        instance=instance,
        overwrite=True)
except Exception as e:
    print(f"Error: {e}")

[Deutsch-Jozsa algorithm](https://quantum.cloud.ibm.com/learning/en/courses/fundamentals-of-quantum-algorithms/quantum-query-algorithms/deutsch-jozsa-algorithm)

In [ ]:
def dj_query(num_qubits):
    # Create a circuit implementing for a query gate for a random function
    # satisfying the promise for the Deutsch-Jozsa problem.
 
    qc = QuantumCircuit(num_qubits + 1)
 
    if np.random.randint(0, 2):
        # Flip output qubit with 50% chance
        qc.x(num_qubits)
    if np.random.randint(0, 2):
        # return constant circuit with 50% chance
        print("Constant function generated")
        return qc.to_gate(label="Oracle")
 
    # Choose half the possible input strings
    on_states = np.random.choice(
        range(2**num_qubits),  # numbers to sample from
        2**num_qubits // 2,  # number of samples
        replace=False,  # makes sure states are only sampled once
    )
 
    def add_cx(qc, bit_string):
        for qubit, bit in enumerate(reversed(bit_string)):
            if bit == "1":
                qc.x(qubit)
        return qc
 
    for state in on_states:
        qc = add_cx(qc, f"{state:0b}")
        qc.mcx(list(range(num_qubits)), num_qubits)
        qc = add_cx(qc, f"{state:0b}")
  
    print("Balanced function generated")
    return qc.to_gate(label="Oracle")

In [ ]:
def compile_circuit(function: QuantumCircuit):
    # Compiles a circuit for use in the Deutsch-Jozsa algorithm.
 
    n = function.num_qubits - 1
    qc = QuantumCircuit(n + 1, n)
    qc.x(n)
    qc.h(range(n + 1))
    qc.compose(function, inplace=True)
    qc.h(range(n))
    
    print("Bloch sphere:")
    display(plot_bloch_multivector(Statevector.from_instruction(qc)))
    
    qc.measure(range(n), range(n))
    return qc

In [ ]:
f = dj_query(3)
print("Oracle:")
display(f.definition.draw(output="mpl", style="iqp"))

qc = compile_circuit(f)

print("Deutsch-Jozsa circuit:")
display(qc.draw(output="mpl", style="iqp"))

If you want to run the circuit on the Aer simulator:

In [ ]:
shots = 1

backend = AerSimulator()
result = backend.run(qc.decompose(), shots=shots, memory=True).result()

measurements = result.get_memory()

If you want to run the circuit on a real quantum computer:

In [ ]:
shots = 500

service = QiskitRuntimeService()
print("Selecting backend...")
backend = service.least_busy(simulator=False, operational=True)
print(f"Running on backend: {backend.name}")

pm = generate_preset_pass_manager(optimization_level=2, backend=backend)
 
transpiled_circuit = pm.run(qc)

sampler = Sampler(backend)

job = sampler.run([transpiled_circuit], shots=shots)

job_id = job.job_id()
print(f"Job sumbitted. ID: {job_id}") # Wait for the job to complete and then retrieve the results

Retrieve the results from the job run on the real quantum computer:

In [ ]:
retrieved_job = service.job(job_id)
current_status = retrieved_job.status()
if current_status == 'DONE':
    result = retrieved_job.result()
    measurements = result[0].data.c.get_bitstrings()
    print("Job completed successfully!")
else:
    print(f"Job is not completed yet. Current status: {current_status}")

In [ ]:
f_type = "balanced" if "1" in measurements[0] else "constant"
print("The function is: " + f_type)

## Deutsch-Jozsa vs Classical Complexity

### 1. Speedup: Exponential Advantage
The Deutsch-Jozsa algorithm determines if a hidden function is **constant** (same output for all inputs) or **balanced** (output is 0 for half and 1 for the other half).
* **Classical Case:** Requires checking more than half of the possible inputs ($2^{n-1} + 1$) to be 100% certain. This is an **exponential** complexity $O(2^n)$.
* **Quantum Case:** Solves the problem in exactly **one single query ($O(1)$)**, regardless of the number of qubits ($n$).



### 2. How it works: Quantum Interference
Instead of checking inputs one by one, the algorithm evaluates all possible inputs in superposition. Through **interference**, the amplitudes of the wrong answers cancel out (destructive interference), while the correct answer is amplified (constructive interference). After only one oracle call, the final measurement provides the answer with 100% certainty.